# Notebook 05: Real Batching Throughput/Latency + Grounded PagedAttention Simulation

`[REAL]` + `[SIMULATION]` Companion to Modules 03 and 06. Real batched generation on the RTX 4060 measuring real throughput/latency across batch sizes (Module 06), then feeding this notebook's own real measured per-request generation lengths into a Module-03-style paged-vs-contiguous allocation simulation.

**Per the signed-off plan:** the PagedAttention piece is explicitly labeled `[SIMULATION]`, not `[REAL]`, even though it uses genuinely real measured inputs -- it is not an actual PagedAttention implementation or a real A/B benchmark against contiguous serving, and this notebook does not claim otherwise.

In [1]:
import time
import math
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16).to(DEVICE)
model.eval()
print(f"Loaded {MODEL_NAME} at FP16 on {DEVICE}")

D:\Study\Prep\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:  47%|████▋     | 136/290 [00:00<00:00, 1343.42it/s]

Loading weights:  93%|█████████▎| 271/290 [00:00<00:00, 1258.80it/s]

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1284.38it/s]

Loaded Qwen/Qwen2.5-0.5B-Instruct at FP16 on cuda


## 1. Real Batched Throughput/Latency Across Batch Sizes

`[REAL]` A real, diverse set of 8 chat-formatted prompts (short, naturally-varied-length real answers), run at real batch sizes 1/2/4/8, measuring real wall-clock batch completion time and real throughput (useful tokens generated per second, counting only up to each real sequence's own first EOS token -- not padding).

In [2]:
PROMPTS = [
    "What is the capital of France? Answer in one short sentence.",
    "Say hello in exactly 3 words.",
    "What is 2+2? Answer with just the number.",
    "Name one primary color.",
    "What is the chemical symbol for water?",
    "Name a planet in our solar system.",
    "What language is spoken in Japan?",
    "Give the opposite of 'hot' in one word.",
]
CHAT_PROMPTS = [tokenizer.apply_chat_template([{"role": "user", "content": p}], tokenize=False,
                                               add_generation_prompt=True) for p in PROMPTS]
MAX_NEW_TOKENS = 80

def real_generated_lengths(output_ids, prompt_len):
    """Real per-sequence useful length: position of first EOS token in the generated region,
    or the full generated length if no real EOS was emitted within MAX_NEW_TOKENS."""
    lengths = []
    for seq in output_ids:
        gen_ids = seq[prompt_len:].tolist()
        eos_pos = next((j + 1 for j, tid in enumerate(gen_ids) if tid == tokenizer.eos_token_id), len(gen_ids))
        lengths.append(eos_pos)
    return lengths

BATCH_SIZES = [1, 2, 4, 8]
throughput_results = []
all_real_lengths = None
for bs in BATCH_SIZES:
    batch_prompts = CHAT_PROMPTS[:bs]
    inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True).to(DEVICE)
    prompt_len = inputs["input_ids"].shape[1]

    torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                              pad_token_id=tokenizer.pad_token_id)
    torch.cuda.synchronize()
    elapsed_s = time.perf_counter() - start

    real_lengths = real_generated_lengths(out, prompt_len)
    total_useful_tokens = sum(real_lengths)
    throughput_tok_per_s = total_useful_tokens / elapsed_s
    per_request_latency_ms = elapsed_s * 1000 / bs
    throughput_results.append({
        "batch_size": bs, "elapsed_s": elapsed_s, "total_useful_tokens": total_useful_tokens,
        "throughput_tok_per_s": throughput_tok_per_s, "per_request_latency_ms": per_request_latency_ms,
    })
    print(f"batch_size={bs}: elapsed={elapsed_s:.2f}s, useful_tokens={total_useful_tokens}, "
          f"throughput={throughput_tok_per_s:.2f} tok/s, per-request latency={per_request_latency_ms:.1f}ms")
    if bs == BATCH_SIZES[-1]:
        all_real_lengths = real_lengths

print(f"\nReal per-request generation lengths at batch_size={BATCH_SIZES[-1]}: {all_real_lengths}")
print("\n(pending real interpretation)")

batch_size=1: elapsed=1.53s, useful_tokens=8, throughput=5.25 tok/s, per-request latency=1525.3ms


batch_size=2: elapsed=0.77s, useful_tokens=18, throughput=23.27 tok/s, per-request latency=386.8ms


batch_size=4: elapsed=0.81s, useful_tokens=27, throughput=33.41 tok/s, per-request latency=202.0ms


batch_size=8: elapsed=6.11s, useful_tokens=200, throughput=32.71 tok/s, per-request latency=764.2ms

Real per-request generation lengths at batch_size=8: [8, 10, 2, 7, 11, 80, 80, 2]

(pending real interpretation)


**Real result — a genuine, unplanned straggler effect:** throughput scaled well from `bs=1` (`5.25 tok/s`) to `bs=4` (`33.41 tok/s`, a real `6.36x` improvement), but then real throughput *dropped slightly* at `bs=8` (`32.71 tok/s`) and real per-request latency *jumped* from `202.0ms` at `bs=4` to `764.2ms` at `bs=8` — a real `278%` increase. The cause is visible directly in the real per-request generation lengths at `bs=8`: `[8, 10, 2, 7, 11, 80, 80, 2]`. Two of the eight real sequences never emitted a real EOS token within the `80`-token budget, forcing the *entire* batch's wall-clock time to stretch to accommodate them — a real, naturally-occurring (not staged) demonstration of exactly the static-batching straggler problem Module 06 describes: the batch can't finish until its slowest real member does, regardless of how short the other real requests were (`min=2`, `max=80`, a real `40x` spread).

## 2. Grounded PagedAttention Simulation (Real Measured Inputs, Simulated Allocation)

`[SIMULATION]` Feeding this notebook's own real measured per-request generation lengths (Section 1's largest batch) into a Module-03-style contiguous-vs-paged allocation simulation. **This is a simulation grounded in real data, not an actual PagedAttention implementation or a real A/B serving benchmark** -- stated explicitly, per the signed-off plan.

In [3]:
MAX_SUPPORTED_LEN = MAX_NEW_TOKENS  # the real max_new_tokens ceiling this batch was run under
BLOCK_SIZE = 8

def contiguous_allocation_waste(lengths, max_len):
    reserved = max_len * len(lengths)
    used = sum(lengths)
    waste = reserved - used
    return {"reserved": reserved, "used": used, "waste": waste, "waste_pct": waste / reserved * 100}

def paged_allocation_waste(lengths, block_size):
    reserved = sum(math.ceil(L / block_size) * block_size for L in lengths)
    used = sum(lengths)
    waste = reserved - used
    return {"reserved": reserved, "used": used, "waste": waste, "waste_pct": waste / reserved * 100}

real_lengths = all_real_lengths
print(f"Real measured generation lengths used as simulation input: {real_lengths}")

contiguous_sim = contiguous_allocation_waste(real_lengths, MAX_SUPPORTED_LEN)
paged_sim = paged_allocation_waste(real_lengths, BLOCK_SIZE)

print(f"\n[SIMULATION] Contiguous: reserved={contiguous_sim['reserved']}, used={contiguous_sim['used']}, "
      f"waste={contiguous_sim['waste']} ({contiguous_sim['waste_pct']:.2f}%)")
print(f"[SIMULATION] Paged (block={BLOCK_SIZE}): reserved={paged_sim['reserved']}, used={paged_sim['used']}, "
      f"waste={paged_sim['waste']} ({paged_sim['waste_pct']:.2f}%)")

print("\n(pending real interpretation)")

Real measured generation lengths used as simulation input: [8, 10, 2, 7, 11, 80, 80, 2]

[SIMULATION] Contiguous: reserved=640, used=200, waste=440 (68.75%)
[SIMULATION] Paged (block=8): reserved=224, used=200, waste=24 (10.71%)

(pending real interpretation)


## 3. Real Interpretation

`[SIMULATION]` Using this notebook's own real, straggler-containing generation lengths (`[8, 10, 2, 7, 11, 80, 80, 2]`) — not synthetic numbers — the simulated contiguous allocation wastes `68.75%` of its reserved memory (`440` of `640` slots), while the simulated paged allocation (block size `8`) wastes only `10.71%` (`24` of `224` slots) — a real-data-grounded `6.42x` waste reduction, closely matching the qualitative pattern Module 03's own (synthetic) worked example demonstrated. **This remains a `[SIMULATION]`**: no actual PagedAttention memory manager or real multi-request serving system ran here — only a Python-level allocation-accounting exercise using genuinely real generation-length inputs, exactly as scoped in the signed-off plan. The straggler pair (`80`, `80`) that caused Section 1's real latency spike are the same two real values driving most of the simulated contiguous waste here — a direct, real link between this notebook's two sections, not a coincidence.